# Model Testing and Production Comparison

This notebook evaluates the final trained model (`best_model.pt`) on the held-out test
set, then documents two deployment optimization paths that were investigated: post-training
dynamic quantization and export to ONNX Runtime.

Dynamic quantization is included here specifically because it was tried and discarded,
it measurably degraded task performance, and that negative result is worth keeping as
part of the record rather than deleting silently. **ONNX Runtime export
(`scripts/export_model.py`) is what actually runs in production**, served through
`scripts/inference.py`. The comparison below is the evidence that motivated that choice.

### Setup

In [1]:
import sys
import os
import copy
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from scripts.dataset import PortfolioDataset, multitask_collate_fn

# Production target is a CPU-only VM evaluating everything
# on CPU here keeps the comparison fair across all three approaches below.
DEVICE = torch.device("cpu")

test_dataset = PortfolioDataset(
    str(PROJECT_ROOT / "data" / "cleaned" / "processed" / "test_processed.jsonl")
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=multitask_collate_fn,
)

print(f"Test samples: {len(test_dataset)}")

Test samples: 235


### Loading the Trained Model

The model is imported directly from `scripts/model.py`.

In [2]:
from scripts.model import MultitaskXLM


model = MultitaskXLM(num_domain_labels=10, num_ner_labels=3)
model.load_state_dict(torch.load(
    PROJECT_ROOT / "models" / "best_model.pt",
    map_location=DEVICE,
))
model.to(DEVICE)
model.eval()

c:\Users\LCELIL\projet-integration\ai\venv\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


MultitaskXLM(
  (encoder): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
           

### Evaluation Function

To avoid duplication we import `eval_step` from `scripts/eval.py` .

In [4]:
from scripts.eval import eval_step

def file_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2)

### Baseline Evaluation (fp32 PyTorch)

In [5]:
baseline_metrics = eval_step(model, test_dataloader, DEVICE)
print("Baseline (unquantized PyTorch):")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.4f}")

Baseline (unquantized PyTorch):
  loss: 0.0894
  domain_micro_f1: 0.8913
  domain_macro_f1: 0.8981
  ner_micro_f1: 0.9740
  ner_macro_f1: 0.9461


### Dynamic Quantization — Investigated and Discarded

Post-training dynamic quantization (`torch.quantization.quantize_dynamic`) converts
the weights of every `nn.Linear` layer to `int8`, aiming to shrink the model and speed
up CPU inference with no retraining required. It was tested as a candidate deployment
optimization before ONNX Runtime was adopted.

The cell below re-runs the exact same test-set evaluation on the quantized model, so
the F1 numbers are directly comparable to the baseline above.

In [6]:
quantized_model = torch.quantization.quantize_dynamic(
    copy.deepcopy(model),
    {nn.Linear},
    dtype=torch.qint8,
)
quantized_model.eval()

quantized_metrics = eval_step(quantized_model, test_dataloader, DEVICE)
print("Dynamically quantized (int8):")
for k, v in quantized_metrics.items():
    print(f"  {k}: {v:.4f}")

# Measure on-disk size without leaving a stray artifact in the repo
with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as tmp:
    torch.save(quantized_model.state_dict(), tmp.name)
    quantized_size_mb = file_size_mb(tmp.name)
os.remove(tmp.name)
print(f"\nQuantized model size: {quantized_size_mb:.1f} MB")

Dynamically quantized (int8):
  loss: 0.1151
  domain_micro_f1: 0.8587
  domain_macro_f1: 0.8638
  ner_micro_f1: 0.9512
  ner_macro_f1: 0.8931

Quantized model size: 816.1 MB


### ONNX Runtime — What Actually Ships to Production

The model deployed to production is exported to ONNX by `scripts/export_model.py` and
served by `scripts/inference.py`. Rather than re-implementing that pipeline here, this
section re-evaluates the exported ONNX graph on the same test set, using the identical
batches from `test_dataloader`, this confirms the exported graph reproduces the
trained model's actual behavior on held-out data, not just that it runs without
crashing.

In [7]:
import onnxruntime as ort

ONNX_MODEL_PATH = PROJECT_ROOT / "models" / "model.onnx"

session = ort.InferenceSession(
    str(ONNX_MODEL_PATH),
    providers=["CPUExecutionProvider"],
)

def onnx_eval_step(session, dataloader):
    all_domain_preds, all_domain_labels = [], []
    all_ner_preds, all_ner_labels = [], []

    for batch in dataloader:
        input_ids = batch["input_ids"].numpy().astype(np.int64)
        attention_mask = batch["attention_mask"].numpy().astype(np.int64)
        domain_labels = batch["domain_labels"].numpy().astype(int)
        ner_labels = batch["ner_labels"].numpy()

        domain_logits, ner_logits = session.run(
            None,
            {"input_ids": input_ids, "attention_mask": attention_mask},
        )

        domain_preds = (1 / (1 + np.exp(-domain_logits)) > 0.5).astype(int)
        all_domain_preds.append(domain_preds)
        all_domain_labels.append(domain_labels)

        ner_preds = np.argmax(ner_logits, axis=-1)
        mask = ner_labels != -100
        all_ner_preds.append(ner_preds[mask])
        all_ner_labels.append(ner_labels[mask])

    all_domain_preds = np.concatenate(all_domain_preds)
    all_domain_labels = np.concatenate(all_domain_labels)
    all_ner_preds = np.concatenate(all_ner_preds)
    all_ner_labels = np.concatenate(all_ner_labels)

    return {
        "domain_micro_f1": f1_score(all_domain_labels, all_domain_preds, average="micro", zero_division=0),
        "domain_macro_f1": f1_score(all_domain_labels, all_domain_preds, average="macro", zero_division=0),
        "ner_micro_f1": f1_score(all_ner_labels, all_ner_preds, average="micro", zero_division=0),
        "ner_macro_f1": f1_score(all_ner_labels, all_ner_preds, average="macro", zero_division=0),
    }

onnx_metrics = onnx_eval_step(session, test_dataloader)
print("ONNX Runtime (production):")
for k, v in onnx_metrics.items():
    print(f"  {k}: {v:.4f}")

ONNX Runtime (production):
  domain_micro_f1: 0.8861
  domain_macro_f1: 0.8940
  ner_micro_f1: 0.9740
  ner_macro_f1: 0.9461


### Summary and Production Decision

Quantization and ONNX Runtime were explored for different reasons: quantization was
tried to shrink the checkpoint itself, while ONNX Runtime was adopted for CPU inference
performance and deployment footprint. The exported graph and the original PyTorch
checkpoint are essentially the same size on disk, the deployment size reduction comes
from the Docker image no longer needing PyTorch and its training-graph dependencies,
not from the model file itself getting smaller. The table below puts F1 and size side
by side for all three approaches anyway, generated directly from the runs above rather
than hardcoded, so the actual size relationship is visible rather than assumed.

In [8]:
comparison = pd.DataFrame({
    "Baseline (fp32)": {
        **{k: v for k, v in baseline_metrics.items() if k != "loss"},
        "size_mb": file_size_mb(PROJECT_ROOT / "models" / "best_model.pt"),
    },
    "Quantized (int8)": {
        **{k: v for k, v in quantized_metrics.items() if k != "loss"},
        "size_mb": quantized_size_mb,
    },
    "ONNX Runtime (production)": {
        **onnx_metrics,
        "size_mb": file_size_mb(PROJECT_ROOT / "models" / "model.onnx"),
    },
}).T

comparison.round(4)

,domain_micro_f1,domain_macro_f1,ner_micro_f1,ner_macro_f1,size_mb
Baseline (fp32),0.8913,0.8981,0.9740,0.9461,1060.7711
Quantized (int8),0.8587,0.8638,0.9512,0.8931,816.1092
ONNX Runtime (production),0.8861,0.8940,0.9740,0.9461,1058.6881


Dynamic quantization reduced model size but degraded task performance enough to
be disqualified for production use, int8 quantization of the linear layers introduces
enough numerical error in the pooled representations to measurably hurt both domain
classification and specifically technology detection on this task ( Even if the F1 scores doesn't show a big difference ). ONNX Runtime, by contrast, preserves full fp32 precision, its
F1 scores should match the baseline almost exactly, while still delivering the CPU
inference speed and reduced dependency footprint (no PyTorch in the deployment
container) that motivated looking at quantization in the first place. That combination
is why ONNX Runtime was adopted instead.

### Sanity Check: Production Inference Path

As a final check, the exact `predict_text` function used in production
(`scripts/inference.py`) is run on a few representative, unseen example descriptions —
confirming the exported ONNX model behaves correctly end-to-end, including technology
normalization, not just on the raw F1 metrics above.

*Note: `load_model()` initializes `TechNormaliser`, which persists its registry to
`data/tech_registry.json` after every call — running this cell will write to that file
if any of the examples below contain a technology name not already in the registry.
Expected behavior in production, worth knowing before re-running this repeatedly.*

In [ ]:
from scripts.inference import load_model, predict_text
from pathlib import Path

_original_cwd = os.getcwd()
try:
    os.chdir(PROJECT_ROOT)

    # Guard against an empty registry file causing json.load() to blow up.
    registry_path = Path("data/tech_registry.json")
    if registry_path.exists() and registry_path.stat().st_size == 0:
        registry_path.write_text("{}", encoding="utf-8")

    session, tokenizer, tech_normaliser = load_model()

    examples = [
        "The Smart Retail Management System (SRMS) is a cloud-based enterprise application developed for medium-sized retail businesses to automate inventory management, sales tracking, customer relationship management, and business analytics.",
        "Developed a multilingual AI system based on Transformer architectures to automatically analyze and classify technical portfolio content. The project combines Named Entity Recognition (NER) and multi-label text classification in a single Multi-Task Learning (MTL) model using XLM-RoBERTa.",
        "Developed a scalable e-commerce platform using React, Next.js, and TypeScript for the frontend with a Node.js and Express backend. Integrated Redis caching and PostgreSQL for performance optimization. Containerized the application with Docker and deployed it on Kubernetes using Helm charts and GitHub Actions CI/CD pipelines. Implemented OAuth2 authentication, JWT session management, and Web Application Firewall protection with Cloudflare.",
        "Designed a smart irrigation system powered by ESP32 microcontrollers and MQTT communication. Sensor data for soil humidity and temperature was transmitted to AWS IoT Core and visualized through a Flutter mobile application. Implemented Bluetooth Low Energy pairing and offline synchronization for remote agricultural environments with unstable connectivity.",
        "Built a real-time anomaly detection pipeline using Python, Kafka, and Spark Structured Streaming, deployed on Kubernetes with Helm charts, monitored via Prometheus and Grafana, models trained with scikit-learn and tracked in MLflow, infrastructure provisioned by Terraform on AWS, with a React dashboard consuming a FastAPI backend secured by OAuth2.",
    ]

    for text in examples:
        result = predict_text(text, session, tokenizer, tech_normaliser)
        print(f"TEXT: {text}\n")
        print(f"  Domains: {result['domains']}")
        print(f"  Technologies (normalized): {result['technologies']}")
        print(f"  Technologies (raw): {result['raw_technologies']}")
        print("\n" + "=" * 80 + "\n")
finally:
    os.chdir(_original_cwd)

TEXT: The Smart Retail Management System (SRMS) is a cloud-based enterprise application developed for medium-sized retail businesses to automate inventory management, sales tracking, customer relationship management, and business analytics.

  Domains: ['Web Backend']
  Technologies (normalized): []
  Technologies (raw): []


TEXT: Developed a multilingual AI system based on Transformer architectures to automatically analyze and classify technical portfolio content. The project combines Named Entity Recognition (NER) and multi-label text classification in a single Multi-Task Learning (MTL) model using XLM-RoBERTa.

  Domains: ['Machine Learning and AI']
  Technologies (normalized): ['XLM-RoBERTa']
  Technologies (raw): ['XLM-RoBERTa']


TEXT: Developed a scalable e-commerce platform using React, Next.js, and TypeScript for the frontend with a Node.js and Express backend. Integrated Redis caching and PostgreSQL for performance optimization. Containerized the application with Docker and 

### Conclusion

This notebook establishes, with test-set evidence rather than assertion, why ONNX
Runtime is the production inference path for this model: dynamic quantization
measurably hurt both domain classification and NER accuracy, while ONNX Runtime
preserves the trained model's numerical behavior (confirmed by the F1 parity with the
fp32 baseline above), runs noticeably faster on plain CPU, and drops the PyTorch
dependency and training-graph overhead from the deployment container. Model size on
disk is essentially unchanged between the fp32 baseline and the ONNX export, the
deployment footprint reduction comes from removing PyTorch and its dependencies from
the image, not from a smaller model file. The exported graph, tokenizer, and technology
registry produced here are the same artifacts consumed by the FastAPI microservice
described in the deployment chapter of the technical report.